# Bayesian neural network: MCMC, vanilla VI, and warm-start VI

This tutorial uses **one probabilistic neural-network model** and fits it in three ways:

1. **MCMC (NUTS):** a sampling-based reference.
2. **Vanilla variational inference (VI):** a mean-field `AutoDiagonalNormal` guide trained with stochastic variational inference (SVI), using NumPyro's default initialization.
3. **Warm-start VI:** the same variational family and ELBO, but initialized using a deterministic neural-network fit, with a smaller learning rate and several stochastic optimization runs.

We first show the behavior of vanilla VI, *including a potentially poor fit*, and only then introduce the warm-start techniques. All three methods use the **same data, network architecture, likelihood, and priors**. Warm-start pretraining changes only the optimizer's starting point; it does not change the Bayesian model.

**What we compare:** the inferred latent (noise-free) function, the posterior predictive distribution (which includes observation noise), the inferred observation-noise scale, and the optimization losses. The `X_test` variable below is an **evaluation grid**, not a labeled holdout test set.

**Dependencies:** `jax`, `numpyro`, `numpy`, and `matplotlib` (Graphviz is optional for the model diagram). The MCMC run and warm-start VI may take some time; execute cells in order.

In [ ]:
from __future__ import annotations

from time import perf_counter

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
from jax.example_libraries import optimizers as jax_optimizers
from numpyro.infer import MCMC, NUTS, Predictive, SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDiagonalNormal
from numpyro.infer.initialization import init_to_value
from numpyro.optim import Adam

print("JAX devices:", jax.devices())

## 1. Generate the synthetic data

The inputs contain polynomial features of a single scalar input. The target function is nonlinear and noisy; we standardize the observed targets as in the original tutorial. The evaluation grid extends beyond the training range, so extrapolation is visible.

In [ ]:
def get_data(
    N: int = 50,
    D_X: int = 3,
    sigma_obs: float = 0.05,
    N_test: int = 500,
    seed: int = 0,
) -> tuple[jax.Array, jax.Array, jax.Array]:
    rng = np.random.default_rng(seed)

    X = jnp.linspace(-1, 1, N)
    X = jnp.power(X[:, np.newaxis], jnp.arange(D_X))

    W = 0.5 * rng.standard_normal(D_X)

    Y = jnp.dot(X, W) + 0.5 * jnp.power(0.5 + X[:, 1], 2.0) * jnp.sin(4.0 * X[:, 1])
    Y += sigma_obs * rng.standard_normal(N)
    Y = Y[:, np.newaxis]
    Y -= jnp.mean(Y)
    Y /= jnp.std(Y)

    X_test = jnp.linspace(-1.3, 1.3, N_test)
    X_test = jnp.power(X_test[:, np.newaxis], jnp.arange(D_X))

    return X, Y, X_test

In [ ]:
N = 100
D_X = 3
D_H = 5

X, Y, X_test = get_data(
    N=N,
    D_X=D_X,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(np.asarray(X[:, 1]), np.asarray(Y[:, 0]), color="black", marker="x")
plt.xlabel("X")
plt.ylabel("Y (standardized)")
plt.title("Synthetic training data")
plt.grid(alpha=0.15)
plt.show()

## 2. Define the common Bayesian neural network

We use two hidden layers, each with `D_H = 5` units and `tanh` nonlinearities. Every weight has a standard Normal prior. Observation precision has a Gamma prior, and the observation model is Normal:

\[
 w_i \sim \mathcal N(0,1),\qquad
 \tau \sim \operatorname{Gamma}(3,1),\qquad
 Y\mid X,w,\tau \sim \mathcal N(f_w(X),\tau^{-1/2}).
\]

The `mu` site exposes the **noise-free network output** for diagnostic plots; it does not change the model's probability distributions. We keep this exact model for MCMC and both VI variants.

In [ ]:
def nonlin(x: jax.Array) -> jax.Array:
    return jnp.tanh(x)


def model(
    X: jax.Array,
    Y: jax.Array | None = None,
    D_H: int = 5,
    D_Y: int = 1,
) -> None:
    N, D_X = X.shape

    w1 = numpyro.sample(
        "w1",
        dist.Normal(
            jnp.zeros((D_X, D_H)),
            jnp.ones((D_X, D_H)),
        ),
    )

    z1 = nonlin(jnp.matmul(X, w1))

    w2 = numpyro.sample(
        "w2",
        dist.Normal(
            jnp.zeros((D_H, D_H)),
            jnp.ones((D_H, D_H)),
        ),
    )

    z2 = nonlin(jnp.matmul(z1, w2))

    w3 = numpyro.sample(
        "w3",
        dist.Normal(
            jnp.zeros((D_H, D_Y)),
            jnp.ones((D_H, D_Y)),
        ),
    )

    z3 = jnp.matmul(z2, w3)
    numpyro.deterministic("mu", z3)  # noiseless network prediction

    prec_obs = numpyro.sample(
        "prec_obs",
        dist.Gamma(3.0, 1.0),
    )

    sigma_obs = 1.0 / jnp.sqrt(prec_obs)

    with numpyro.plate("data", N):
        numpyro.sample(
            "Y",
            dist.Normal(z3, sigma_obs).to_event(1),
            obs=Y,
        )

**Optional visualization:** model diagrams require a working Graphviz installation. The remaining cells do not depend on it.

In [ ]:
try:
    display(numpyro.render_model(
        model,
        model_kwargs={"X": X, "Y": Y, "D_H": D_H},
        render_distributions=True,
    ))
except Exception as exc:
    print(f"Skipping optional model diagram: {exc}")

## 3. Shared posterior-predictive helpers

For each method, draw from the fitted weight posterior (or its VI approximation), then run **the same model** on the same evaluation grid. We report both:

- **Latent-function interval:** uncertainty about the underlying neural-network function `mu`.
- **Posterior predictive interval:** uncertainty about a new observation `Y`, including observation noise.

Each interval spans the 5th to the 95th percentile (a 90% interval).

In [ ]:
NUM_POSTERIOR_DRAWS = 2000


def posterior_summary(samples, rng_key):
    """Return equal-footing latent and noisy predictions on the shared X_test grid."""
    sites = Predictive(
        model, posterior_samples=samples, return_sites=["mu", "Y"]
    )(rng_key, X=X_test, Y=None, D_H=D_H)

    mu_draws = np.asarray(sites["mu"][..., 0])
    y_draws = np.asarray(sites["Y"][..., 0])
    return {
        "mu_mean": mu_draws.mean(axis=0),
        "mu_interval": np.percentile(mu_draws, [5, 95], axis=0),
        "y_mean": y_draws.mean(axis=0),
        "y_interval": np.percentile(y_draws, [5, 95], axis=0),
    }


def plot_single_fit(summary, name, color):
    plt.figure(figsize=(9, 4.5))
    plt.scatter(X[:, 1], Y[:, 0], marker="x", color="black", s=24,
                label="Training observations")
    plt.fill_between(X_test[:, 1], *summary["y_interval"],
                     color=color, alpha=0.22, label="90% predictive interval")
    plt.plot(X_test[:, 1], summary["mu_mean"],
             color=color, linewidth=2, label="Posterior mean of latent function")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(name)
    plt.legend(loc="best")
    plt.show()


def observation_noise_median(samples):
    return float(np.median(1.0 / np.sqrt(np.asarray(samples["prec_obs"]))))


def latent_training_mse(samples, rng_key):
    """In-sample diagnostic, NOT a held-out prediction metric."""
    mu_train = Predictive(
        model, posterior_samples=samples, return_sites=["mu"]
    )(rng_key, X=X, Y=None, D_H=D_H)["mu"][..., 0]
    return float(np.mean((np.asarray(mu_train).mean(axis=0) - np.asarray(Y[:, 0])) ** 2))

## 4. Reference: MCMC using NUTS

Run the original NUTS configuration: 1,000 warmup iterations and 2,000 posterior draws from one chain. A single chain is sufficient for this demonstration, but **not enough for thorough convergence validation**; a formal analysis should use multiple chains and diagnostics such as $\hat{R}$ and effective sample size.

In [ ]:
mcmc_start = perf_counter()
mcmc = MCMC(NUTS(model), num_warmup=1000, num_samples=2000, num_chains=1)
mcmc.run(jax.random.PRNGKey(0), X=X, Y=Y, D_H=D_H)
mcmc_samples = mcmc.get_samples()
jax.block_until_ready(mcmc_samples["w1"])
mcmc_elapsed = perf_counter() - mcmc_start
mcmc.print_summary()
print(f"MCMC wall time (including compilation/warmup): {mcmc_elapsed:.1f} s")

In [ ]:
mcmc_summary = posterior_summary(mcmc_samples, jax.random.PRNGKey(10))
plot_single_fit(mcmc_summary, "MCMC (NUTS): posterior predictive", "tab:blue")

## 5. Vanilla variational inference (no pretraining)

First run **unmodified, vanilla mean-field VI** using `AutoDiagonalNormal(model)`, Adam with learning rate `0.01`, and 5,000 SVI iterations. This is the baseline implementation from the original VI notebook. VI minimizes the negative evidence lower bound (negative ELBO).

The guide assumes a factorized Gaussian in the **unconstrained latent space**. Since the BNN has a highly nonconvex parameterization, optimization can converge to a solution with a nearly flat mean function and inflated observation noise. We do not correct this result before showing it.

In [ ]:
vanilla_guide = AutoDiagonalNormal(model)
vanilla_svi = SVI(model, vanilla_guide, Adam(step_size=0.01), loss=Trace_ELBO())

vanilla_start = perf_counter()
vanilla_result = vanilla_svi.run(
    jax.random.PRNGKey(1), 5000, X=X, Y=Y, D_H=D_H
)
jax.block_until_ready(vanilla_result.losses)
vanilla_elapsed = perf_counter() - vanilla_start

print(f"Vanilla VI wall time: {vanilla_elapsed:.1f} s")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(np.asarray(vanilla_result.losses), linewidth=0.9, color="tab:orange")
plt.xlabel("SVI iteration")
plt.ylabel("Negative ELBO")
plt.title("Vanilla VI: optimization loss")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
vanilla_samples = vanilla_guide.sample_posterior(
    jax.random.PRNGKey(11),
    vanilla_result.params,
    sample_shape=(NUM_POSTERIOR_DRAWS,),
)
vanilla_summary = posterior_summary(vanilla_samples, jax.random.PRNGKey(12))
plot_single_fit(vanilla_summary, "Vanilla VI: posterior predictive", "tab:orange")

print(f"Vanilla VI median sigma_obs: {observation_noise_median(vanilla_samples):.3f}")
print("Compare against MCMC median sigma_obs:",
      f"{observation_noise_median(mcmc_samples):.3f}")

### Vanilla VI versus MCMC

Plot these two fits **before** using any warm-start technique. If the vanilla VI curve is almost flat, that is an optimization outcome, not evidence that the posterior-predictive sampling code is broken. A falling negative ELBO does not guarantee accurate predictions.

In [ ]:
def plot_against_mcmc(other_summary, other_label, other_color, title):
    plt.figure(figsize=(10, 5.5))
    plt.scatter(X[:, 1], Y[:, 0], marker="x", color="black", s=24,
                label="Training observations")
    plt.fill_between(X_test[:, 1], *mcmc_summary["y_interval"],
                     color="tab:blue", alpha=0.17, label="MCMC 90% predictive interval")
    plt.plot(X_test[:, 1], mcmc_summary["mu_mean"],
             color="tab:blue", lw=2, label="MCMC latent mean")
    plt.fill_between(X_test[:, 1], *other_summary["y_interval"],
                     color=other_color, alpha=0.18,
                     label=f"{other_label} 90% predictive interval")
    plt.plot(X_test[:, 1], other_summary["mu_mean"],
             color=other_color, lw=2, label=f"{other_label} latent mean")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(title)
    plt.legend(loc="best")
    plt.show()


plot_against_mcmc(vanilla_summary, "Vanilla VI", "tab:orange",
                  "MCMC versus vanilla VI (before warm start)")

## 6. Improve VI using deterministic pretraining

Only **after** observing vanilla VI do we introduce the optimization techniques from the improved notebook:

1. Fit the same neural-network forward function deterministically on the **training data** (five random initializations, 3,000 Adam steps each, a small weight penalty).
2. Use the weights from the best deterministic fit to initialize VI via `init_to_value`; initialize observation precision using that fit's residual error. These are **initial values only**: all latent variables remain learnable during VI.
3. Reduce the SVI learning rate from `0.01` to `0.003`, increase the number of iterations to 8,000, and compare three stochastic SVI runs.
4. Select the SVI run with the smallest average negative ELBO over its final 500 steps, *not* according to how well its plotted curve matches MCMC.

The three SVI runs below have **different random seeds but share the same pretrained starting weights**. They are *not* independent initializations in weight space. The five deterministic pretraining attempts *are* independent random initializations.

These optimization aids cost additional computation. The probability model, mean-field guide family, and target ELBO stay unchanged.

In [ ]:
def mlp_forward(weights, inputs):
    w1, w2, w3 = weights
    return jnp.tanh(jnp.tanh(inputs @ w1) @ w2) @ w3


def pretrain_loss(weights, inputs, targets):
    mse = jnp.mean((mlp_forward(weights, inputs) - targets) ** 2)
    weight_penalty = sum(jnp.sum(w ** 2) for w in weights)
    return mse + 5e-4 * weight_penalty


opt_init, opt_update, get_weights = jax_optimizers.adam(0.01)


@jax.jit
def pretrain_step(step, opt_state, inputs, targets):
    weights = get_weights(opt_state)
    grads = jax.grad(pretrain_loss)(weights, inputs, targets)
    return opt_update(step, grads, opt_state)


pretrain_start = perf_counter()
pretrain_key = jax.random.PRNGKey(100)
best_pretrain_loss = float("inf")
pretrain_losses = []

for restart in range(5):
    pretrain_key, init_key = jax.random.split(pretrain_key)
    init_keys = jax.random.split(init_key, 3)
    initial_weights = (
        0.7 * jax.random.normal(init_keys[0], (D_X, D_H)),
        0.7 * jax.random.normal(init_keys[1], (D_H, D_H)),
        0.7 * jax.random.normal(init_keys[2], (D_H, 1)),
    )

    opt_state = opt_init(initial_weights)
    for step in range(3000):
        opt_state = pretrain_step(step, opt_state, X, Y)

    trained_weights = get_weights(opt_state)
    current_loss = float(pretrain_loss(trained_weights, X, Y))
    pretrain_losses.append(current_loss)
    if current_loss < best_pretrain_loss:
        best_pretrain_loss = current_loss
        pretrained_weights = trained_weights

jax.block_until_ready(pretrained_weights[0])
pretrain_elapsed = perf_counter() - pretrain_start
pretrain_mean = mlp_forward(pretrained_weights, X)
pretrain_mse = float(jnp.mean((pretrain_mean - Y) ** 2))

print("Deterministic training objectives:", np.round(pretrain_losses, 4))
print(f"Selected fit: training MSE = {pretrain_mse:.4f}")
print(f"Pretraining wall time: {pretrain_elapsed:.1f} s")

plt.figure(figsize=(9, 4))
plt.scatter(X[:, 1], Y[:, 0], marker="x", color="black",
            label="Training observations")
plt.plot(X[:, 1], pretrain_mean[:, 0], color="tab:green", lw=2,
         label="Selected deterministic fit")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Deterministic pretraining: initialization check")
plt.legend()
plt.show()

## 7. Warm-start VI: optimize the **same** ELBO

We initialize each `AutoDiagonalNormal` guide near the pretrained weights. The initial value for `prec_obs` is estimated from the deterministic residual scale and clipped to avoid pathological extremes. This affects **only where optimization begins**, not the Gamma prior or likelihood. The guide still learns all its parameters from the original Bayesian ELBO.

This section can take longer than vanilla VI. For a quick trial, reduce `warm_vi_steps` or `warm_vi_runs`, and restore the defaults for the final comparison.

In [ ]:
initial_sigma = float(np.clip(np.sqrt(pretrain_mse), 0.15, 0.7))
initial_values = {
    "w1": pretrained_weights[0],
    "w2": pretrained_weights[1],
    "w3": pretrained_weights[2],
    "prec_obs": jnp.array(1.0 / initial_sigma**2),
}
print(f"Initial observation-noise std (not fixed): {initial_sigma:.3f}")

warm_vi_steps = 8000
warm_vi_runs = 3
warm_trials = []
warm_vi_start = perf_counter()

for run_id in range(warm_vi_runs):
    # Identical pretrained center; different SVI randomness across runs.
    trial_guide = AutoDiagonalNormal(
        model,
        init_loc_fn=init_to_value(values=initial_values),
        init_scale=0.05,
    )
    trial_svi = SVI(
        model, trial_guide, Adam(step_size=0.003), loss=Trace_ELBO()
    )
    result = trial_svi.run(
        jax.random.PRNGKey(1000 + run_id), warm_vi_steps,
        X=X, Y=Y, D_H=D_H,
    )
    jax.block_until_ready(result.losses)
    tail_loss = float(jnp.mean(result.losses[-500:]))
    warm_trials.append({"guide": trial_guide, "result": result, "score": tail_loss})
    print(f"Warm VI run {run_id + 1}: mean final-500 negative ELBO = {tail_loss:.2f}")

warm_vi_elapsed = perf_counter() - warm_vi_start
best_warm = min(warm_trials, key=lambda trial: trial["score"])
warm_guide = best_warm["guide"]
warm_result = best_warm["result"]
print(f"Selected run score: {best_warm['score']:.2f}")
print(f"Warm SVI wall time (all runs): {warm_vi_elapsed:.1f} s")

In [ ]:
plt.figure(figsize=(9, 4))
for run_id, trial in enumerate(warm_trials):
    plt.plot(np.asarray(trial["result"].losses), lw=0.9, alpha=0.75,
             label=f"Run {run_id + 1}")
plt.xlabel("SVI iteration")
plt.ylabel("Negative ELBO")
plt.title("Warm-start VI: stochastic optimization runs (same initialization)")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
warm_samples = warm_guide.sample_posterior(
    jax.random.PRNGKey(2000), warm_result.params,
    sample_shape=(NUM_POSTERIOR_DRAWS,),
)
warm_summary = posterior_summary(warm_samples, jax.random.PRNGKey(2001))
plot_single_fit(warm_summary, "Warm-start VI: posterior predictive", "tab:green")

plot_against_mcmc(warm_summary, "Warm-start VI", "tab:green",
                  "MCMC versus warm-start VI")

## 8. Compare all three methods

In the first plot we compare the **latent posterior means** (without adding observation noise). In the second plot we compare the posterior predictive intervals side by side against MCMC. These are visual diagnostics, not formal tests of posterior accuracy.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(X[:, 1], Y[:, 0], marker="x", color="black", s=24,
            label="Training observations")
plt.plot(X_test[:, 1], mcmc_summary["mu_mean"], color="tab:blue", lw=2,
         label="MCMC (NUTS)")
plt.plot(X_test[:, 1], vanilla_summary["mu_mean"], color="tab:orange", lw=2,
         label="Vanilla VI")
plt.plot(X_test[:, 1], warm_summary["mu_mean"], color="tab:green", lw=2,
         label="Warm-start VI")
plt.xlabel("X")
plt.ylabel("Latent mean / observations")
plt.title("Three inference approaches: mean latent function")
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

for ax, candidate, name, color in [
    (axes[0], vanilla_summary, "Vanilla VI", "tab:orange"),
    (axes[1], warm_summary, "Warm-start VI", "tab:green"),
]:
    ax.scatter(X[:, 1], Y[:, 0], marker="x", color="black", s=18,
               label="Training data")
    ax.fill_between(X_test[:, 1], *mcmc_summary["y_interval"],
                    color="tab:blue", alpha=0.16, label="MCMC 90% predictive")
    ax.plot(X_test[:, 1], mcmc_summary["mu_mean"],
            color="tab:blue", lw=2, label="MCMC latent mean")
    ax.fill_between(X_test[:, 1], *candidate["y_interval"],
                    color=color, alpha=0.20, label=f"{name} 90% predictive")
    ax.plot(X_test[:, 1], candidate["mu_mean"], color=color, lw=2,
            label=f"{name} latent mean")
    ax.set_title(f"MCMC vs. {name}")
    ax.set_xlabel("X")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Y")
fig.suptitle("Same model and evaluation grid: predictive distributions")
fig.tight_layout()
plt.show()

## 9. Diagnose function uncertainty versus observation noise

A broad predictive interval can arise because the network function is uncertain, because observation noise is large, or both. The following plot explicitly separates latent-function intervals from noisy-observation intervals, using each VI fit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True, sharey=True)
for ax, summary, name, color in [
    (axes[0], vanilla_summary, "Vanilla VI", "tab:orange"),
    (axes[1], warm_summary, "Warm-start VI", "tab:green"),
]:
    ax.scatter(X[:, 1], Y[:, 0], marker="x", color="black", s=19,
               label="Training observations")
    ax.fill_between(X_test[:, 1], *summary["y_interval"],
                    color=color, alpha=0.17, label="90% noisy predictive")
    ax.fill_between(X_test[:, 1], *summary["mu_interval"],
                    color="tab:purple", alpha=0.23, label="90% latent-function")
    ax.plot(X_test[:, 1], summary["mu_mean"], color=color, lw=2,
            label="Mean latent function")
    ax.set_title(name)
    ax.set_xlabel("X")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Y")
fig.tight_layout()
plt.show()

In [ ]:
diagnostics = {
    "MCMC (NUTS)": (mcmc_samples, jax.random.PRNGKey(301)),
    "Vanilla VI": (vanilla_samples, jax.random.PRNGKey(302)),
    "Warm-start VI": (warm_samples, jax.random.PRNGKey(303)),
}

print(f"{'Method':<18} {'Median sigma_obs':>17} {'Train latent MSE':>18}")
print("-" * 56)
for name, (samples, diagnostic_key) in diagnostics.items():
    print(f"{name:<18} {observation_noise_median(samples):>17.3f} "
          f"{latent_training_mse(samples, diagnostic_key):>18.4f}")

print("\nElapsed wall time (seconds; includes JAX compilation):")
print(f"  MCMC:                  {mcmc_elapsed:8.1f}")
print(f"  Vanilla VI:            {vanilla_elapsed:8.1f}")
print(f"  Warm-start pretraining:{pretrain_elapsed:8.1f}")
print(f"  Warm-start SVI:        {warm_vi_elapsed:8.1f}")
print(f"  Warm-start total:      {pretrain_elapsed + warm_vi_elapsed:8.1f}")

## 10. Interpretation and methodological limitations

- **Same statistical target:** all methods use the same priors, likelihood, network shape, and observed dataset. The deterministic fit **only initializes** warm-start VI; it does not become an additional prior or likelihood term.
- **Vanilla versus tuned VI:** the point is to show the baseline *before* applying initialization and optimizer techniques. The warm-start improvement does not prove that mean-field VI can represent the true BNN posterior accurately.
- **ELBO is not predictive accuracy:** a falling negative ELBO shows optimization progress, but it does not guarantee a good approximate posterior or a well-calibrated predictive distribution.
- **Different uncertainty sources:** intervals for `mu` describe uncertainty in the latent function, whereas intervals for `Y` additionally include measurement/observation noise.
- **Fair timing:** warm-start runtime includes all five pretraining starts and all three VI runs. MCMC runtime includes warmup. Algorithmic efficiency claims should account for these costs and for hyperparameter tuning.
- **Not an out-of-sample accuracy benchmark:** `X_test` is only an evaluation grid (there are no `Y_test` labels). The reported MSE is **in-sample**; generalization and uncertainty calibration require a genuinely held-out dataset.
- **MCMC caveat:** the one-chain demonstration cannot establish convergence. Use multiple chains and examine \(\hat R\), ESS, and trace plots for a rigorous reference.
- **VI caveat:** `AutoDiagonalNormal` ignores posterior correlations in the unconstrained space and cannot capture multiple separated modes. Better initialization improves optimization but not the variational family's expressive power. A richer guide (e.g. `AutoLowRankMultivariateNormal`) would test this limitation.

**Main lesson:** a good optimization strategy for VI is standard practice and is not inherently "cheating". However, it should be reported explicitly, and the original vanilla VI run should remain visible so readers can distinguish the benefits of initialization from differences in inference methodology.